In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("SmartHomeEnergyTracker") \
    .getOrCreate()

In [2]:
from google.colab import files

uploaded = files.upload()

Saving energy_usage.csv to energy_usage.csv


In [3]:
df = spark.read.csv(
    "energy_usage.csv",
    header=True,
    inferSchema=True
)

df.show()

+---------+------------+-------+-------------------+----------+
|device_id| device_name|room_id|          timestamp|energy_kwh|
+---------+------------+-------+-------------------+----------+
|      101|          AC|      1|2026-06-01 08:00:00|       5.2|
|      102|          TV|      1|2026-06-01 09:00:00|       1.5|
|      103|         Fan|      2|2026-06-01 10:00:00|       2.1|
|      104|Refrigerator|      3|2026-06-01 19:00:00|       4.3|
|      101|          AC|      1|2026-06-01 20:00:00|       6.0|
|      102|          TV|      1|2026-06-01 21:00:00|       1.7|
|      103|         Fan|      2|2026-06-01 14:00:00|       2.8|
|      104|Refrigerator|      3|2026-06-01 22:00:00|       4.7|
+---------+------------+-------+-------------------+----------+



In [4]:
df = df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [5]:
df = df.withColumn(
    "hour",
    hour("timestamp")
)

df = df.withColumn(
    "usage_type",
    when(
        (col("hour") >= 18) &
        (col("hour") <= 22),
        "Peak"
    ).otherwise("Off-Peak")
)

df.show()

+---------+------------+-------+-------------------+----------+----+----------+
|device_id| device_name|room_id|          timestamp|energy_kwh|hour|usage_type|
+---------+------------+-------+-------------------+----------+----+----------+
|      101|          AC|      1|2026-06-01 08:00:00|       5.2|   8|  Off-Peak|
|      102|          TV|      1|2026-06-01 09:00:00|       1.5|   9|  Off-Peak|
|      103|         Fan|      2|2026-06-01 10:00:00|       2.1|  10|  Off-Peak|
|      104|Refrigerator|      3|2026-06-01 19:00:00|       4.3|  19|      Peak|
|      101|          AC|      1|2026-06-01 20:00:00|       6.0|  20|      Peak|
|      102|          TV|      1|2026-06-01 21:00:00|       1.7|  21|      Peak|
|      103|         Fan|      2|2026-06-01 14:00:00|       2.8|  14|  Off-Peak|
|      104|Refrigerator|      3|2026-06-01 22:00:00|       4.7|  22|      Peak|
+---------+------------+-------+-------------------+----------+----+----------+



In [6]:
peak_usage = (
    df.groupBy(
        "device_id",
        "device_name",
        "usage_type"
    )
    .agg(
        round(
            sum("energy_kwh"),2
        ).alias("total_energy")
    )
)

peak_usage.show()

+---------+------------+----------+------------+
|device_id| device_name|usage_type|total_energy|
+---------+------------+----------+------------+
|      102|          TV|  Off-Peak|         1.5|
|      103|         Fan|  Off-Peak|         4.9|
|      101|          AC|      Peak|         6.0|
|      102|          TV|      Peak|         1.7|
|      101|          AC|  Off-Peak|         5.2|
|      104|Refrigerator|      Peak|         9.0|
+---------+------------+----------+------------+



In [7]:
top_devices = (
    df.groupBy(
        "device_id",
        "device_name"
    )
    .agg(
        round(
            sum("energy_kwh"),2
        ).alias("total_energy")
    )
    .orderBy(
        col("total_energy").desc()
    )
)

top_devices.show()

+---------+------------+------------+
|device_id| device_name|total_energy|
+---------+------------+------------+
|      101|          AC|        11.2|
|      104|Refrigerator|         9.0|
|      103|         Fan|         4.9|
|      102|          TV|         3.2|
+---------+------------+------------+



In [8]:
heavy_devices = top_devices.filter(
    col("total_energy") > 5
)

heavy_devices.show()

+---------+------------+------------+
|device_id| device_name|total_energy|
+---------+------------+------------+
|      101|          AC|        11.2|
|      104|Refrigerator|         9.0|
+---------+------------+------------+



In [9]:
top_devices.write \
.mode("overwrite") \
.option("header", True) \
.csv("top_devices_output")

In [10]:
!zip -r top_devices_output.zip top_devices_output

  adding: top_devices_output/ (stored 0%)
  adding: top_devices_output/part-00000-a798c058-55cf-4170-b1c7-4db82a25f384-c000.csv (deflated 8%)
  adding: top_devices_output/._SUCCESS.crc (stored 0%)
  adding: top_devices_output/.part-00000-a798c058-55cf-4170-b1c7-4db82a25f384-c000.csv.crc (stored 0%)
  adding: top_devices_output/_SUCCESS (stored 0%)


In [11]:
from google.colab import files
files.download("top_devices_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>